# Generar dos submissions desde la v10.4 ya ejecutada

Esta notebook **no reentrena**. Reutiliza exclusivamente las dos predicciones creadas
por la corrida v10.4 causal de ~17 millones de filas y genera:

1. `100% v3`.
2. `90% v3 + 10% Ridge-LightGBM residual`.

Tiempo esperado: segundos o menos de un minuto.


## 1) Ambiente y ubicación del bucket

In [ ]:
from pathlib import Path
import csv
import os
import numpy as np
import pandas as pd


def resolver_bucket():
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser().resolve()
    for candidata in (
        Path.home() / "buckets" / "b1",
        Path("/content/buckets/b1"),
        Path("/home/ds/buckets/b1"),
    ):
        if candidata.is_dir():
            return candidata
    raise RuntimeError("No encontré el bucket. Definí LABO3_BUCKET.")


BUCKET = resolver_bucket()
OUT = BUCKET / "submissions_v10_4_pesos_finales"
OUT.mkdir(parents=True, exist_ok=True)
print("BUCKET:", BUCKET)
print("SALIDA:", OUT)


## 2) Localizar los dos componentes de la misma corrida

In [ ]:
PATH_V3 = (
    BUCKET
    / "pipe_v10_4_fast_17m_corregida_v3"
    / "submission_x1.0.csv"
)

PATRON_RESIDUAL = (
    "exp_residuo/"
    "*base-lineal_esq-E_lineal_mas_lgbm*"
    "v10_4_fast_v4_5semillas_leakage_safe/"
    "submission_202002.csv"
)
candidatos_residuo = sorted(
    BUCKET.glob(PATRON_RESIDUAL),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not PATH_V3.exists():
    raise FileNotFoundError(
        f"No existe la predicción v3 de la corrida corregida: {PATH_V3}"
    )
if not candidatos_residuo:
    raise FileNotFoundError(
        "No encontré la predicción Ridge-residuo. Patrón buscado: "
        f"{BUCKET / PATRON_RESIDUAL}"
    )

PATH_RESIDUAL = candidatos_residuo[0]
print("Componente v3:", PATH_V3)
print("Componente Ridge-residuo:", PATH_RESIDUAL)
if len(candidatos_residuo) > 1:
    print("Se eligió el archivo más reciente entre", len(candidatos_residuo), "candidatos")


## 3) Validar y alinear componentes

In [ ]:
def leer_prediccion(path, nombre):
    df = pd.read_csv(path)
    df.columns = (
        df.columns.astype(str)
          .str.replace("\ufeff", "", regex=False)
          .str.strip()
    )
    unnamed = [c for c in df.columns if c.lower().startswith("unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    if df.columns.tolist() != ["product_id", "tn"]:
        raise RuntimeError(
            f"{nombre}: columnas {df.columns.tolist()}, esperaba ['product_id', 'tn']"
        )
    df["product_id"] = pd.to_numeric(df["product_id"], errors="raise").astype("int64")
    df["tn"] = pd.to_numeric(df["tn"], errors="raise").astype("float64")
    if df["product_id"].duplicated().any():
        raise RuntimeError(f"{nombre}: existen product_id duplicados")
    if df["tn"].isna().any() or not np.isfinite(df["tn"]).all():
        raise RuntimeError(f"{nombre}: existen tn nulas o infinitas")
    if (df["tn"] < 0).any():
        raise RuntimeError(f"{nombre}: existen tn negativas")
    return df.sort_values("product_id").reset_index(drop=True)


v3 = leer_prediccion(PATH_V3, "v3").rename(columns={"tn": "tn_v3"})
residual = leer_prediccion(PATH_RESIDUAL, "Ridge-residuo").rename(
    columns={"tn": "tn_residual"}
)

ambos = v3.merge(residual, on="product_id", how="outer", validate="one_to_one", indicator=True)
if not (ambos["_merge"] == "both").all():
    raise RuntimeError(ambos["_merge"].value_counts().to_string())
ambos = ambos.drop(columns="_merge")

print("Productos alineados:", len(ambos))
print("TN v3:", ambos["tn_v3"].sum())
print("TN Ridge-residuo:", ambos["tn_residual"].sum())
print("Correlación:", ambos[["tn_v3", "tn_residual"]].corr().iloc[0, 1])


## 4) Crear los dos CSV finales

In [ ]:
def guardar_submission(nombre, peso_v3, peso_residual):
    if abs(peso_v3 + peso_residual - 1.0) > 1e-12:
        raise ValueError("Los pesos no suman 1")
    salida = ambos[["product_id"]].copy()
    salida["tn"] = np.maximum(
        peso_v3 * ambos["tn_v3"].to_numpy()
        + peso_residual * ambos["tn_residual"].to_numpy(),
        0.0,
    )
    path = OUT / nombre
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["product_id", "tn"])
        for product_id, tn in salida.itertuples(index=False, name=None):
            w.writerow([int(product_id), format(float(tn), ".10f")])
    check = pd.read_csv(path)
    if check.columns.tolist() != ["product_id", "tn"]:
        raise RuntimeError(f"{nombre}: encabezado incorrecto")
    if len(check) != len(ambos) or check["product_id"].duplicated().any():
        raise RuntimeError(f"{nombre}: filas o identificadores incorrectos")
    if check["tn"].isna().any() or not np.isfinite(check["tn"]).all():
        raise RuntimeError(f"{nombre}: predicciones inválidas")
    print(f"{nombre}: filas={len(check)}, tn_total={check['tn'].sum():,.4f}")
    return path


PATH_100 = guardar_submission(
    "submission_v10_4_v3_100_residual_00.csv", 1.00, 0.00
)
PATH_9010 = guardar_submission(
    "submission_v10_4_v3_90_residual_10.csv", 0.90, 0.10
)

print("\n✅ DOS SUBMISSIONS GENERADAS Y VALIDADAS")
print("1:", PATH_100)
print("2:", PATH_9010)


## 5) Envío

El envío automático queda desactivado. Descargá y subí manualmente los dos CSV.

Orden recomendado:

1. `submission_v10_4_v3_100_residual_00.csv`
2. `submission_v10_4_v3_90_residual_10.csv`
